In [ ]:
%pip install mne
%pip install EMD-signal
%pip install statsmodels
%pip install onnxruntime
%pip install mne-qt-browser PyQt5

In [2]:
import mne
import os
import glob
import numpy as np
from PyEMD import EMD
from mne_icalabel import label_components
import statsmodels.api as sm
import matplotlib.pyplot as plt

mne.viz.set_browser_backend('qt')

Using qt as 2D backend.


In [3]:
def apply_emd_baseline(epoch_data):
    """
    Aplica EMD canal por canal em um único segmento (epoch).
    Remove o último IMF (intrinsic mode function), que representa a linha de base.
    """
    emd = EMD()
    corrected_data = np.zeros_like(epoch_data)
    
    for ch in range(epoch_data.shape[0]):
        # O PyEMD espera um array 1D (sinal de um canal em um epoch)
        imfs = emd.emd(epoch_data[ch])
        
        if imfs.shape[0] > 1:
            # reconstrói o sinal somando todos os IMFs, exceto o último (baseline)
            corrected_data[ch] = np.sum(imfs[:-1], axis=0)
        else:
            corrected_data[ch] = epoch_data[ch]
            
    return corrected_data

def reject_by_cooks_distance(epochs_data, base_name="Paciente", plot_folder=None):
    n_epochs = epochs_data.shape[0]
    if n_epochs <= 1:
        return np.array([])

    y = np.var(epochs_data, axis=(1, 2))
    x = np.ones(n_epochs)
    model = sm.OLS(y.astype(float), x.astype(float)).fit()
    influence = model.get_influence()
    cooks = influence.cooks_distance[0] 
    
    mean_cooks = np.mean(cooks)
    cutoff = 4 * mean_cooks
    bad_epochs_idx = np.where(cooks > cutoff)[0]

    if plot_folder:
        os.makedirs(plot_folder, exist_ok=True)
        
        plt.figure(figsize=(10, 6))
        
        plt.scatter(range(n_epochs), cooks, color='#1f77b4', alpha=0.7, label='Segmentos Mantidos')
        
        # segmentos rejeitados em vermelho
        if len(bad_epochs_idx) > 0:
            plt.scatter(bad_epochs_idx, cooks[bad_epochs_idx], color='red', label='Outliers (Rejeitados)')
        
        # linha de corte
        plt.axhline(y=cutoff, color='red', linestyle='--', label=f'Threshold (4x Média)')
        
        plt.xlabel('Número da Epoch (Segmento de 5s)', fontsize=12)
        plt.ylabel("Distância de Cook", fontsize=12)
        plt.title(f'Diagnóstico de Outliers de Variância - {base_name}', fontsize=14)
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.6)

        plot_path = os.path.join(plot_folder, f"{base_name}_cooks_distance_ec.png")
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close()

    return bad_epochs_idx

In [4]:
input_folder = 'data'
output_folder = 'cleaned_ec'
plots_folder = 'plots_outliers'

os.makedirs(output_folder, exist_ok=True)
os.makedirs(plots_folder, exist_ok=True)

gdf_files = glob.glob(os.path.join(input_folder, '*.gdf'))

if not gdf_files:
    print(f"Nenhum arquivo .gdf encontrado na pasta '{input_folder}'.")
else:
    print(f"Encontrados {len(gdf_files)} arquivos para processar.\n")

for file in gdf_files:
    base_name = os.path.splitext(os.path.basename(file))[0]
    print(f"Processando: {base_name}")
    
    raw = mne.io.read_raw_gdf(file, preload=True, verbose=False)
    raw_proc = raw.copy()
    
    montage = mne.channels.make_standard_montage('standard_1020')
    raw_proc.set_montage(montage, match_case=False, on_missing='warn')
    
    # filtragem
    raw_proc.filter(l_freq=0.1, h_freq=100.0, verbose=False) 
    raw_proc.notch_filter(freqs=60.0, verbose=False)
    raw_proc.set_eeg_reference('average', projection=True, verbose=False)
    raw_proc.apply_proj()
    
    # mne-icalabe
    ica = mne.preprocessing.ICA(n_components=15, method='infomax', fit_params=dict(extended=True), random_state=97, max_iter='auto')
    ica.fit(raw_proc, verbose=False)
    
    ic_labels = label_components(raw_proc, ica, method='iclabel')
    labels = ic_labels['labels']
    
    # remove componentes olho, músculo, coração ou ruído
    bads_ica = [idx for idx, label in enumerate(labels) if label in ['eye', 'muscle', 'heart', 'channel noise', 'line noise']]
    ica.exclude = bads_ica
    print(f"Componentes removidos pelo ICLabel: {bads_ica} (Rótulos: {[labels[i] for i in bads_ica]})")
    ica.apply(raw_proc, verbose=False)

    # recorte do sinal (Olhos Fechados: 305s até o fim)
    raw_ec = raw_proc.copy().crop(tmin=305.0, tmax=None)

    # segmentação em janelas de 5s
    epochs = mne.make_fixed_length_epochs(raw_ec, duration=5.0, preload=True, verbose=False)

    # índice cronológico real de cada época no ID do evento
    for i in range(len(epochs.events)):
        epochs.events[i, 2] = i
    
    # identificação de Outliers (> 100 µV)
    # O MNE avalia a amplitude peak-to-peak. Se um outlier é > 100 µV absoluto, 
    # o pico a pico máximo permitido seria 200 µV (200e-6 Volts).
    reject_criteria = dict(eeg=200e-6) 
    epochs.drop_bad(reject=reject_criteria, verbose=False)
    print(f"Epochs restantes após remoção de amplitude (> 100 µV): {len(epochs)}")

    # pula o paciente se tudo for deletado
    if len(epochs) == 0:
        print(f"Todos os segmentos de {base_name} eram ruído e foram removidos.\n")
        continue

    # identificação de outliers (Cook's Distance)
    epochs_data = epochs.get_data() # Formato: (n_epochs, n_channels, n_times)
    bad_cooks_idx = reject_by_cooks_distance(epochs_data, base_name=base_name, plot_folder=plots_folder)
    if len(bad_cooks_idx) > 0:
        print(f"Epochs removidos pela Distância de Cook: {bad_cooks_idx}")
        epochs.drop(bad_cooks_idx, verbose=False)
        epochs_data = epochs.get_data() # Atualiza os dados após remoção

    # pula o paciente se a Distância de Cook apagar o que restou
    if len(epochs) == 0:
        print(f"Todos os segmentos de {base_name} foram removidos pela Distância de Cook. Paciente pulado.\n")
        continue

    # Baseline Correction com EMD
    print("Aplicando EMD para correção de linha de base")
    for ep_idx in range(epochs_data.shape[0]):
        epochs_data[ep_idx] = apply_emd_baseline(epochs_data[ep_idx])
    
    # Atualiza o objeto Epochs com os dados corrigidos pelo EMD
    epochs_corrected = mne.EpochsArray(epochs_data, epochs.info, events=epochs.events, tmin=epochs.tmin, verbose=False)

    save_filename = f"{base_name}_preproc_ec-epo.fif"
    save_path = os.path.join(output_folder, save_filename)
    
    epochs_corrected.save(save_path, overwrite=True, verbose=False)
    print(f"Arquivo salvo: {save_filename}\n")

print("Processamento de todos os arquivos concluído")

Encontrados 37 arquivos para processar.

Processando: ID0
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...


C:\Users\Renan\AppData\Local\Temp\ipykernel_2756\1816837331.py:35: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_proc, ica, method='iclabel')
C:\Users\Renan\AppData\Local\Temp\ipykernel_2756\1816837331.py:35: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_proc, ica, method='iclabel')


Componentes removidos pelo ICLabel: [] (Rótulos: [])
Epochs restantes após remoção de amplitude (> 100 µV): 54
Epochs removidos pela Distância de Cook: [ 3 34 36]
Aplicando EMD para correção de linha de base
Arquivo salvo: ID0_preproc_ec-epo.fif

Processando: ID1
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...


C:\Users\Renan\AppData\Local\Temp\ipykernel_2756\1816837331.py:35: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_proc, ica, method='iclabel')
C:\Users\Renan\AppData\Local\Temp\ipykernel_2756\1816837331.py:35: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_proc, ica, method='iclabel')


Componentes removidos pelo ICLabel: [] (Rótulos: [])
Epochs restantes após remoção de amplitude (> 100 µV): 57
Epochs removidos pela Distância de Cook: [51 56]
Aplicando EMD para correção de linha de base
Arquivo salvo: ID1_preproc_ec-epo.fif

Processando: ID10
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...


...

Componentes removidos pelo ICLabel: [] (Rótulos: [])
Epochs restantes após remoção de amplitude (> 100 µV): 40
Epochs removidos pela Distância de Cook: [21 22 28 39]
Aplicando EMD para correção de linha de base
Arquivo salvo: ID7_preproc_ec-epo.fif

Processando: ID8
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...


C:\Users\Renan\AppData\Local\Temp\ipykernel_2756\1816837331.py:35: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_proc, ica, method='iclabel')
C:\Users\Renan\AppData\Local\Temp\ipykernel_2756\1816837331.py:35: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_proc, ica, method='iclabel')


Componentes removidos pelo ICLabel: [] (Rótulos: [])
Epochs restantes após remoção de amplitude (> 100 µV): 33
Epochs removidos pela Distância de Cook: [28 29]
Aplicando EMD para correção de linha de base
Arquivo salvo: ID8_preproc_ec-epo.fif

Processando: ID9
Created an SSP operator (subspace dimension = 1)
1 projection items activated
SSP projectors applied...


C:\Users\Renan\AppData\Local\Temp\ipykernel_2756\1816837331.py:35: RuntimeWarning: The provided Raw instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_proc, ica, method='iclabel')
C:\Users\Renan\AppData\Local\Temp\ipykernel_2756\1816837331.py:35: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_proc, ica, method='iclabel')


Componentes removidos pelo ICLabel: [] (Rótulos: [])
Epochs restantes após remoção de amplitude (> 100 µV): 57
Epochs removidos pela Distância de Cook: [ 2 29]
Aplicando EMD para correção de linha de base
Arquivo salvo: ID9_preproc_ec-epo.fif

Processamento de todos os arquivos concluído
